Analise exploratório CLA- OMNIBOX

**Objetivo:** Padronização dos dados para o formato '.parquet', exploração e limpeza dos dados. 

**Motor de Processamento:** DuckDB (Processamento Analítico em Memória)

**Visualização:** Pandas e Seaborn (Apenas para agregações e amostras)

### Etapa 1 - Setup e Configurações Iniciais
Importação de bilbiotecas, setup e conexão com o DuckDB e configuração de pastas de trabalho.

In [1]:
# Criar o kernel no Poetry:  
# Garante que o modulo externo seja sempre recarregado 
%load_ext autoreload
%autoreload 2

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from pathlib import Path
import os 
import shutil
import cla_omnibox_tratamento_funcoes as omni_val
import json
import ipywidgets as widgets
from tqdm import tqdm

import sys 

print(10)
# Configurações visuais padronizadas para os gráficos e tabelas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
sns.set_theme(style="whitegrid")

# 1. Inicializando a conexão com o DuckDB (em memória). 
con = duckdb.connect()

# 2. Otimizações de Hardware 
# Limitação do uso de memória RAM para evitar problemas. 
con.execute("PRAGMA memory_limit='6GB'") 

#  Limitação do numero de threads
con.execute("PRAGMA threads=8") 

print("Bibliotecas carregadas e motor DuckDB configurado com sucesso!")

# Configuração das pastas de trabalho (se não existirem serão criadas)

DIR_BASE_APP = Path.cwd() 
DIR_BASE = DIR_BASE_APP.parents[0]
PATH_CONFIG = DIR_BASE / "config"
FILE_JSON_CONFIG = "cla_omnibox_schema_dados.json"
caminho_completo_json = PATH_CONFIG / FILE_JSON_CONFIG
DIR_RAW = DIR_BASE / "dados" / "raw"
DIR_BRONZE = DIR_BASE / "dados" / "bronze"
DIR_PRATA = DIR_BASE / "dados" /  "prata"
DIR_GA = f"{DIR_BRONZE}/google_analytics/*.parquet" 
print(f"Pastas e caminhos de configuração definidos com sucesso!") 

10
Bibliotecas carregadas e motor DuckDB configurado com sucesso!
Pastas e caminhos de configuração definidos com sucesso!


### Etapa 2 - Carga da estrutura de dados 
Carregar json com o esquema de dados a ser validade e alguns parametros de validação. 

In [2]:

print(f"Carregando a estrutura das tabelas (schema)")
try: 

    caminho_completo_json = PATH_CONFIG / FILE_JSON_CONFIG
    with open(caminho_completo_json, 'r', encoding='utf-8') as f:
            config_mestre = json.load(f)

    tabelas_mapeadas = [t['tabela'] for t in config_mestre['esquema']]
    print(f"Tabelas configuradas no esquema: {tabelas_mapeadas}")

    print(f"Sucesso: JSON mestre carregado a partir de {caminho_completo_json}")

except FileNotFoundError:
    print(f"Erro: O arquivo {FILE_JSON_CONFIG} não foi encontrado no caminho {PATH_CONFIG}")
except json.JSONDecodeError:
    print(f"Erro: O arquivo {FILE_JSON_CONFIG} não está em um formato JSON válido.")
except Exception as e:
    print(f"Ocorreu um erro inesperado: {e}")

Carregando a estrutura das tabelas (schema)
Tabelas configuradas no esquema: ['category', 'sub_category', 'products', 'customers', 'orders', 'order_items', 'mkt_campaigns', 'mkt_actions', 'mkt_spend', 'google_analytics']
Sucesso: JSON mestre carregado a partir de c:\gitHub\cla-omnibox\config\cla_omnibox_schema_dados.json


### Etapa 3  - Mapeamento dos arquivos fisicos (parquet) no DuckDB
Criação de **VIEWS** no DuckDB para mapear os arquivos parquet da camada Bronze. 

In [ ]:
# --- Etapa 4: Mapeamento de Metadados e Criacao de Views (Camada Bronze) ---

print("Iniciando mapeamento de metadados para criacao de views...\n")
views_criadas = []
print("inicio") 
for entrada in config_mestre['esquema']:
    tabela_nome = entrada['tabela']
    view_name = tabela_nome 
    caminho_str = None
    origem_log = ""
    
    busca_arquivo = list(DIR_BRONZE.rglob(f"{tabela_nome}.parquet"))
    busca_pasta = [p for p in DIR_BRONZE.rglob(tabela_nome) if p.is_dir()]
    
    if busca_arquivo:
        caminho_str = busca_arquivo[0].as_posix()
        origem_log = busca_arquivo[0].name
    elif busca_pasta:
        pasta_encontrada = busca_pasta[0]
        if list(pasta_encontrada.glob("*.parquet")):
            caminho_str = f"{pasta_encontrada.as_posix()}/*.parquet"
            origem_log = f"Pasta particionada: {pasta_encontrada.name}/*.parquet"
        
    if caminho_str:

        query_view = f"CREATE OR REPLACE VIEW {view_name} AS SELECT * FROM read_parquet('{caminho_str}', union_by_name=true);"
        
        con.execute(query_view)
        views_criadas.append(view_name)
        print(f"View '{view_name}' mapeada com sucesso -> {origem_log}")
    else:
        print(f"Alerta: Arquivo unico ou pasta particionada para '{tabela_nome}' nao encontrados na Bronze.")

if views_criadas:
    print(f"\nSucesso! {len(views_criadas)} views foram disponibilizadas no DuckDB.")

Iniciando mapeamento de metadados para criacao de views...

inicio
View 'category' mapeada com sucesso -> category.parquet
View 'sub_category' mapeada com sucesso -> sub_category.parquet
View 'products' mapeada com sucesso -> products.parquet
View 'customers' mapeada com sucesso -> customers.parquet
View 'orders' mapeada com sucesso -> orders.parquet
View 'order_items' mapeada com sucesso -> order_items.parquet
View 'mkt_campaigns' mapeada com sucesso -> mkt_campaigns.parquet
View 'mkt_actions' mapeada com sucesso -> mkt_actions.parquet
View 'mkt_spend' mapeada com sucesso -> mkt_spend.parquet
View 'google_analytics' mapeada com sucesso -> Pasta particionada: google_analytics/*.parquet

Sucesso! 10 views foram disponibilizadas no DuckDB.


### Etapa 4 - Inspeção de Schema e Volumetria
Verificacao de volumetria e estrutura das tabelas 

In [ ]:
# --- Etapa 5: Análise de Volumetria e Validação de Schema (Camada Bronze) ---

print("Calculando volumetria e validando o schema das tabelas...")

dados_volumetria = []

for entrada in config_mestre['esquema']:
    tabela_nome = entrada['tabela']
    
    # Extrai a lista de nomes das colunas esperadas a partir do JSON Mestre
    campos_esperados = [campo['nome'] for campo in entrada['campos']]
    qtd_colunas_esperadas = len(campos_esperados)
    
    try:
        # 1. Consulta o número total de linhas
        query_linhas = f"SELECT COUNT(*) FROM {tabela_nome}"
        qtd_linhas = con.execute(query_linhas).fetchone()[0]
        
        # 2. Consulta os nomes das colunas físicas mapeadas pelo DuckDB
        query_colunas = f"SELECT column_name FROM information_schema.columns WHERE table_name = '{tabela_nome}'"
        campos_reais = [linha[0] for linha in con.execute(query_colunas).fetchall()]
        qtd_colunas_reais = len(campos_reais)
        
        # 3. Validação de Schema usando Sets (Conjuntos)
        set_esperados = set(campos_esperados)
        set_reais = set(campos_reais)
        
        colunas_faltantes = set_esperados - set_reais # Estão no JSON, mas não no arquivo
        colunas_sobrando = set_reais - set_esperados  # Estão no arquivo, mas não no JSON
        
        # 4. Avalia o status e monta os detalhes
        if not colunas_faltantes and not colunas_sobrando:
            status_schema = "OK"
            detalhes = "-"
        else:
            status_schema = "Divergente"
            lista_detalhes = []
            if colunas_faltantes:
                lista_detalhes.append(f"Falta no arquivo: {', '.join(colunas_faltantes)}")
            if colunas_sobrando:
                lista_detalhes.append(f"Não mapeado no JSON: {', '.join(colunas_sobrando)}")
            detalhes = " | ".join(lista_detalhes)
            
        dados_volumetria.append({
            "Tabela (View)": tabela_nome,
            "Total de Linhas": qtd_linhas,
            "Colunas no Arquivo": qtd_colunas_reais,
            "Colunas no JSON": qtd_colunas_esperadas,
            "Status do Schema": status_schema,
            "Detalhes da Divergência": detalhes
        })
        
    except Exception:
        # Captura o erro caso a View não tenha sido criada na Etapa 4
        dados_volumetria.append({
            "Tabela (View)": tabela_nome,
            "Total de Linhas": "Não mapeada",
            "Colunas no Arquivo": "-",
            "Colunas no JSON": qtd_colunas_esperadas,
            "Status do Schema": "Pendente",
            "Detalhes da Divergência": "Arquivo não encontrado na Bronze"
        })

# Converte para DataFrame do Pandas para uma visualização elegante no Notebook
df_volumetria = pd.DataFrame(dados_volumetria)

# Exibe a tabela formatada
display(df_volumetria)

### Etapa 5 - Validação de Nulos e Brancos
Verifica integridade de campos que não aceitam valores brancos e nulos. 

In [ ]:
# --- Etapa 6: Execução da Validação de Nulos e Brancos ---

print("Iniciando auditoria de nulos e brancos...\n")

lista_resultados_df = []

# Itera sobre todas as tabelas do nosso JSON Mestre
for entrada in config_mestre['esquema']:
    tabela_nome = entrada['tabela']
    campos = entrada['campos']
    
    try:
        # Chama a função do nosso arquivo externo
        df_resultado_tabela = omni_val.validar_nulos_e_brancos(con, tabela_nome, campos)
   
        lista_resultados_df.append(df_resultado_tabela)
        
    except duckdb.CatalogException:
        print(f"Tabela/View '{tabela_nome}' não encontrada no DuckDB. Pulando validação.")
    except Exception as e:
        print(f"Erro ao validar a tabela '{tabela_nome}': {e}")

# Concatena os resultados de todas as tabelas em um único DataFrame
if lista_resultados_df:
    df_auditoria_final = pd.concat(lista_resultados_df, ignore_index=True)
    
    # Exibe o relatório final
    display(df_auditoria_final)
else:
    print("Nenhuma validação pôde ser executada.")

### Etapa 6 - Validação de Tipos de dados

In [ ]:
print("Iniciando auditoria de tipagem de dados...\n")

lista_resultados_tipos_df = []

# Itera sobre todas as tabelas do nosso JSON Mestre
for entrada in config_mestre['esquema']:
    tabela_nome = entrada['tabela']
    campos = entrada['campos']
    
    try:
        # Chama a NOVA função do nosso arquivo externo
        df_resultado_tipo = omni_val.validar_tipagem_dados(con, tabela_nome, campos)
        lista_resultados_tipos_df.append(df_resultado_tipo)
        
    except duckdb.CatalogException:
        print(f"Tabela/View '{tabela_nome}' não encontrada no DuckDB. Pulando validação.")
    except Exception as e:
        print(f"Erro ao validar os tipos da tabela '{tabela_nome}': {e}")

# Concatena e exibe os resultados
if lista_resultados_tipos_df:
    df_auditoria_tipos = pd.concat(lista_resultados_tipos_df, ignore_index=True)
    display(df_auditoria_tipos)
else:
    print("Nenhuma validação de tipos pôde ser executada.")

### Etapa 7 - Validação de valores unicos

In [ ]:
print("Iniciando auditoria de Valores Únicos...\n")

lista_resultados_unicos_df = []

# Itera sobre todas as tabelas do nosso JSON Mestre
for entrada in config_mestre['esquema']:
    tabela_nome = entrada['tabela']
    campos = entrada['campos']
    
    try:
        # Chama a função de unicidade do nosso módulo externo
        df_resultado_unico = omni_val.validar_valores_unicos(con, tabela_nome, campos)
        lista_resultados_unicos_df.append(df_resultado_unico)
        
    except duckdb.CatalogException:
        print(f"Tabela/View '{tabela_nome}' não encontrada no DuckDB. Pulando validação.")
    except Exception as e:
        print(f"Erro ao validar a unicidade da tabela '{tabela_nome}': {e}")

# Concatena e exibe os resultados
if lista_resultados_unicos_df:
    df_auditoria_unicos = pd.concat(lista_resultados_unicos_df, ignore_index=True)
    display(df_auditoria_unicos)
else:
    print("Nenhuma validação de valores únicos pôde ser executada.")

### Etapa 8 - Validação de regras numéricas (zeros e negativos) 

In [ ]:

print("Iniciando auditoria de Qualidade de Dados: Zeros e Negativos...\n")

lista_resultados_num_df = []

# Itera sobre todas as tabelas do nosso JSON Mestre
for entrada in config_mestre['esquema']:
    tabela_nome = entrada['tabela']
    campos = entrada['campos']
    
    try:
        # Chama a função de validação numérica
        df_resultado_num = omni_val.validar_regras_numericas(con, tabela_nome, campos)
        lista_resultados_num_df.append(df_resultado_num)
        
    except duckdb.CatalogException:
        print(f"⚠ Tabela/View '{tabela_nome}' não encontrada no DuckDB. Pulando validação.")
    except Exception as e:
        print(f"Erro ao validar as regras numéricas da tabela '{tabela_nome}': {e}")

# Concatena e exibe os resultados
if lista_resultados_num_df:
    df_auditoria_num = pd.concat(lista_resultados_num_df, ignore_index=True)
    display(df_auditoria_num)
else:
    print("Nenhuma validação numérica pôde ser executada.")

    

### Etapa 9 - Validações de formato (via REGEX) 

In [ ]:
print("Iniciando auditoria de Qualidade formatos (padrões Regex)...\n")

lista_resultados_regex_df = []

# Itera sobre todas as tabelas do nosso JSON Mestre
for entrada in config_mestre['esquema']:
    tabela_nome = entrada['tabela']
    campos = entrada['campos']
    
    try:
        # Chama a função de validação de regex
        df_resultado_regex = omni_val.validar_regex(con, tabela_nome, campos)
        lista_resultados_regex_df.append(df_resultado_regex)
        
    except duckdb.CatalogException:
        print(f"Tabela/View '{tabela_nome}' não encontrada no DuckDB. Pulando validação.")
    except Exception as e:
        print(f"Erro ao validar os formatos regex da tabela '{tabela_nome}': {e}")

# Concatena e exibe os resultados
if lista_resultados_regex_df:
    df_auditoria_regex = pd.concat(lista_resultados_regex_df, ignore_index=True)
    display(df_auditoria_regex)
else:
    print("Nenhuma validação de Regex pôde ser executada.")

### Etapa 10 - Validações de lista de valores 

In [ ]:
print("Iniciando auditoria de Qualidade de Dados: Lista de Valores de Domínio...\n")

lista_resultados_valores_df = []

# Itera sobre todas as tabelas do nosso JSON Mestre
for entrada in config_mestre['esquema']:
    tabela_nome = entrada['tabela']
    campos = entrada['campos']
    
    try:
        # Chama a função de validação de listas
        df_resultado_valores = omni_val.validar_lista_valores(con, tabela_nome, campos)
        lista_resultados_valores_df.append(df_resultado_valores)
        
    except duckdb.CatalogException:
        print(f"Tabela/View '{tabela_nome}' não encontrada no DuckDB. Pulando validação.")
    except Exception as e:
        print(f"Erro ao validar a lista de valores da tabela '{tabela_nome}': {e}")

# Concatena e exibe os resultados
if lista_resultados_valores_df:
    df_auditoria_valores = pd.concat(lista_resultados_valores_df, ignore_index=True)
    display(df_auditoria_valores)
else:
    print("Nenhuma validação de lista de valores pôde ser executada.")

### Etapa 11 Validações de integridade de Chaves primárias (PK) 

In [ ]:
print("Iniciando auditoria de integridade de Chaves Primárias (Simples e Compostas)...\n")

lista_resultados_pk_df = []

for entrada in config_mestre['esquema']:
    tabela_nome = entrada['tabela']
    campos = entrada['campos']
    
    try:
        # Chama a função de validação de PK
        df_resultado_pk = omni_val.validar_chave_primaria(con, tabela_nome, campos)
        lista_resultados_pk_df.append(df_resultado_pk)
        
    except duckdb.CatalogException:
        print(f"Tabela/View '{tabela_nome}' não encontrada no DuckDB. Pulando validação.")
    except Exception as e:
        print(f"Erro ao validar a PK da tabela '{tabela_nome}': {e}")

# Concatena e exibe os resultados
if lista_resultados_pk_df:
    df_auditoria_pk = pd.concat(lista_resultados_pk_df, ignore_index=True)
    display(df_auditoria_pk)
else:
    print("Nenhuma validação de PK pôde ser executada.")

### Etapa 12 - Validações de integridade de Chaves estrangeiras (FK) 

In [ ]:

print("Iniciando auditoria de integridade de Chaves Estrangeiras...\n")

lista_resultados_fk_df = []

for entrada in config_mestre['esquema']:
    tabela_nome = entrada['tabela']
    campos = entrada['campos']
    
    try:
        # Chama a função de validação de FK
        df_resultado_fk = omni_val.validar_chaves_estrangeiras(con, tabela_nome, campos)
        lista_resultados_fk_df.append(df_resultado_fk)
        
    except duckdb.CatalogException as e:
        print(f"Erro de Catálogo: Certifique-se de que todas as tabelas relacionadas foram carregadas. Detalhe: {e}")
    except Exception as e:
        print(f"Erro ao validar FK da tabela '{tabela_nome}': {e}")

# Concatena e exibe os resultados
if lista_resultados_fk_df:
    df_auditoria_fk = pd.concat(lista_resultados_fk_df, ignore_index=True)
    display(df_auditoria_fk)
else:
    print("Nenhuma validação de FK pôde ser executada.")

### 13 - Validação de regra de negocio: Consistencia do total da venda

In [ ]:
def gerar_descricao_estatistica(con: duckdb.DuckDBPyConnection, nome_tabela: str) -> None:
    """
    Gera e exibe o resumo estatístico (numérico e categórico) de uma tabela.
    Ideal para identificar dispersão, outliers e frequência de categorias.
    """
    print(f"\n=== Resumo Estatístico: Tabela {nome_tabela.upper()} ===")
    
    # Carrega os dados da tabela para um DataFrame Pandas
    # Nota: Para tabelas gigantescas, pode-se usar LIMIT ou amostragem
    df = con.execute(f"SELECT * FROM {nome_tabela}").to_df()
    
    # 1. Resumo Numérico (Contagem, Média, Std, Min, Max, Percentis)
    desc_num = df.describe()
    if not desc_num.empty:
        print("\n[Dados Numéricos]")
        display(desc_num)
    
    # 2. Resumo Categórico (Contagem, Únicos, Top, Frequência)
    # include=['O'] foca em objetos/strings
    desc_cat = df.describe(include=['O'])
    if not desc_cat.empty:
        print("\n[Dados Categóricos / Texto]")
        display(desc_cat)
        
    print("-" * 50)
print("Iniciando auditoria de Regras de Negocio Customizadas...\n")

resultados_negocio = []

# 1. Definicao da Regra: Consistencia de Total da Venda
nome_regra_venda = "Consistencia de Total da Venda (Itens vs Cabecalho)"

query_venda_itens = """
    WITH SomaItens AS (
        SELECT 
            order_id, 
            SUM(price * quantity) as valor_bruto_itens
        FROM order_items
        GROUP BY order_id
    ),
    Auditoria AS (
        SELECT 
            v.order_id,
            v.gross_amount as valor_registrado,
            (i.valor_bruto_itens - COALESCE(v.discount, 0)) as valor_calculado
        FROM orders v
        JOIN SomaItens i ON v.order_id = i.order_id
        WHERE v.order_status = 'ÇOMPLETED'
    )
    SELECT COUNT(*) 
    FROM Auditoria 
    WHERE ROUND(valor_registrado, 2) <> ROUND(valor_calculado, 2)
"""

# Chamada da funcao generica
df_regra_venda = omni_val.validar_regra_negocio_sql(con, nome_regra_venda, query_venda_itens)
resultados_negocio.append(df_regra_venda)

# Consolidacao e exibicao
if resultados_negocio:
    df_auditoria_negocio = pd.concat(resultados_negocio, ignore_index=True)
    display(df_auditoria_negocio)
else:
    print("Nenhuma regra de negocio foi validada.")

### 14 - Descrições dos dados 

In [ ]:
# --- Etapa 4: Descrição dos Dados (Estatística Descritiva) ---
# Lista de tabelas que desejamos analisar
tabelas_para_analise = ['orders', 'order_items', 'customers'] 

print("Iniciando análise descritiva das tabelas...")

for tabela in tabelas_para_analise:
    try:
        # Chamada da nova função do seu arquivo de funções auxiliares
        omni_val.gerar_descricao_estatistica(con, tabela)
    except Exception as e:
        print(f"Erro ao analisar tabela {tabela}: {e}")